# Grokking Curriculum: From-Scratch Replication

A 2-layer transformer trained from random init on modular arithmetic, under the same
five curriculum orderings as the GPT-2 fine-tuning experiments.

**Why this exists**
1. ~15 min/run instead of ~3 h, so the full 5 x 3 seed grid finishes in one session.
2. Matches the setup in Power et al. (2022) and Nanda et al. (2023).
3. Tests the weight-norm claim properly. Nanda et al. actually report weight
   norm DECREASING, so the paper's current "negative result" is a mis-citation.
   Cell 10b computes the Gini coefficient of Fourier norms instead, which is
   the progress measure they really propose.
4. Deterministic seeding, so no repeat of the 3,250-vs-3,500 same-seed spread.

**Run order:** every cell top to bottom. Cells 1-5 are setup and take under a minute.
Cell 6 is the grid (leave it running). Cells 7+ are analysis.

**Runtime:** T4 GPU.

In [ ]:
# ── 1. Setup, determinism, Drive ────────────────────────────────────────
import os
# must be set before torch touches CUBLAS
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import json, math, random, time
from dataclasses import dataclass, asdict, field
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/grokking_scratch'
os.makedirs(DRIVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU')

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Drive space check: each run writes ~4 MB, full grid ~60 MB
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print('Drive free: %.1f GB' % (free / 1e9))
if free < 2e8:
    print('WARNING: under 200 MB free. Delete old checkpoint folders first.')
print('Saving to:', DRIVE_DIR)


In [ ]:
# ── 2. Model: minimal decoder-only transformer ──────────────────────────
# Manual attention (no flash/SDPA) so results are bit-reproducible.

class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        att = att.masked_fill(mask, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(y)

class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = Attention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model, bias=False),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model, bias=False),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GrokkTransformer(nn.Module):
    # Sequence is [a, op, b, eq] -> predict c at the final position.
    # Vocab: 0..p-1 are numbers, p is 'op', p+1 is '='.
    def __init__(self, p, d_model=128, n_heads=4, n_layers=2, seq_len=4):
        super().__init__()
        self.p = p
        self.embed = nn.Embedding(p + 2, d_model)
        self.pos = nn.Parameter(torch.randn(seq_len, d_model) * 0.02)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, p, bias=False)

    def forward(self, x):
        h = self.embed(x) + self.pos[:x.shape[1]]
        for blk in self.blocks:
            h = blk(h)
        return self.unembed(self.ln_f(h))[:, -1, :]

    def number_embeddings(self):
        # (p, d_model). .copy() is REQUIRED: on CPU .numpy() shares
        # storage with the tensor, so without it every logged snapshot
        # becomes a live view of the final weights.
        return self.embed.weight[:self.p].detach().cpu().numpy().copy()

_m = GrokkTransformer(97)
print('params: %s' % format(sum(p.numel() for p in _m.parameters()), ','))
del _m


In [ ]:
# ── 3. Data and curriculum orderings ────────────────────────────────────
# apply_curriculum is identical in logic to the GPT-2 experiments.

@dataclass
class Config:
    modulus: int = 97
    operation: str = 'add'
    train_fraction: float = 0.5
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    lr: float = 1e-3
    weight_decay: float = 1.0
    batch_size: int = 512
    max_steps: int = 30000
    warmup_steps: int = 100
    eval_every: int = 250
    embed_every: int = 500
    curriculum: str = 'random'
    seed: int = 0
    out_dir: str = ''

def make_pairs(cfg):
    p = cfg.modulus
    out = []
    for a in range(p):
        for b in range(p):
            c = (a + b) % p if cfg.operation == 'add' else (a * b) % p
            out.append((a, b, c))
    return out

def split_data(pairs, frac, seed):
    rng = random.Random(seed)
    shuf = list(pairs)
    rng.shuffle(shuf)
    n = int(len(shuf) * frac)
    return shuf[:n], shuf[n:]

def _wrap_difficulty(triple, cfg):
    a, b, _ = triple
    raw = (a + b) if cfg.operation == 'add' else (a * b)
    return 1 if raw >= cfg.modulus else 0

def apply_curriculum(pairs, curriculum, cfg):
    if curriculum == 'random':
        return list(pairs)
    if curriculum in ('easy_hard', 'hard_easy'):
        s = sorted(pairs, key=lambda t: _wrap_difficulty(t, cfg))
        return s[::-1] if curriculum == 'hard_easy' else s
    if curriculum == 'residue_blocks':
        blocks = {}
        for t in pairs:
            blocks.setdefault(t[2], []).append(t)
        out = []
        for c in sorted(blocks):
            out.extend(blocks[c])
        return out
    if curriculum == 'balanced_mini_epoch':
        blocks = {}
        for t in pairs:
            blocks.setdefault(t[2], []).append(t)
        rng = random.Random(cfg.seed)
        for c in blocks:
            rng.shuffle(blocks[c])
        out = []
        iters = {c: iter(v) for c, v in sorted(blocks.items())}
        active = sorted(blocks.keys())
        while active:
            nxt = []
            for c in active:
                try:
                    out.append(next(iters[c]))
                    nxt.append(c)
                except StopIteration:
                    pass
            active = nxt
        return out
    raise ValueError('unknown curriculum: ' + str(curriculum))

def to_tensors(pairs, cfg):
    p = cfg.modulus
    OP, EQ = p, p + 1
    x = torch.tensor([[a, OP, b, EQ] for a, b, _ in pairs], dtype=torch.long)
    y = torch.tensor([c for _, _, c in pairs], dtype=torch.long)
    return x, y

CURRICULA = ['random', 'easy_hard', 'hard_easy', 'residue_blocks', 'balanced_mini_epoch']

# sanity: every ordering must be a permutation of the input
_cfg = Config(modulus=13)
_tr, _va = split_data(make_pairs(_cfg), 0.5, 0)
for _c in CURRICULA:
    _o = apply_curriculum(_tr, _c, _cfg)
    assert sorted(_o) == sorted(_tr), _c + ' is not a permutation'
    print('%-22s ok  first 3: %s' % (_c, _o[:3]))
print('\nall orderings valid')


In [ ]:
# ── 4. Training loop ────────────────────────────────────────────────────
# Logs weight norm every eval, embeddings every embed_every steps.

def weight_norm(model):
    tot = 0.0
    for p_ in model.parameters():
        tot += float(p_.detach().pow(2).sum().item())
    return math.sqrt(tot)

@torch.no_grad()
def accuracy(model, x, y, bs=2048):
    model.eval()
    correct = 0
    for i in range(0, len(x), bs):
        logits = model(x[i:i+bs].to(device))
        correct += int((logits.argmax(-1) == y[i:i+bs].to(device)).sum().item())
    model.train()
    return correct / len(x)

def train(cfg, verbose_every=2500):
    set_all_seeds(cfg.seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    pairs = make_pairs(cfg)
    train_pairs, val_pairs = split_data(pairs, cfg.train_fraction, cfg.seed)
    ordered = apply_curriculum(train_pairs, cfg.curriculum, cfg)

    xtr, ytr = to_tensors(ordered, cfg)
    xva, yva = to_tensors(val_pairs, cfg)
    xtr_eval, ytr_eval = to_tensors(train_pairs, cfg)

    shuffle = (cfg.curriculum == 'random')
    gen = torch.Generator()
    gen.manual_seed(cfg.seed)
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=cfg.batch_size,
                        shuffle=shuffle, drop_last=True, num_workers=0, generator=gen)

    model = GrokkTransformer(cfg.modulus, cfg.d_model, cfg.n_heads, cfg.n_layers).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                            weight_decay=cfg.weight_decay, betas=(0.9, 0.98))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / cfg.warmup_steps))

    history, embed_steps, embed_mats = [], [], []

    def loop(dl):
        while True:
            for b in dl:
                yield b

    it = loop(loader)
    t0 = time.time()
    model.train()

    for step in range(cfg.max_steps + 1):
        xb, yb = next(it)
        logits = model(xb.to(device))
        loss = F.cross_entropy(logits, yb.to(device))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()

        if step % cfg.eval_every == 0:
            tr_acc = accuracy(model, xtr_eval, ytr_eval)
            va_acc = accuracy(model, xva, yva)
            history.append({
                'step': step,
                'train_loss': float(loss.item()),
                'train_acc': tr_acc,
                'val_acc': va_acc,
                'gen_gap': tr_acc - va_acc,
                'weight_norm': weight_norm(model),
            })
            if step % verbose_every == 0:
                print('  step %6d | loss %.4f | train %.3f | val %.3f | wnorm %.1f'
                      % (step, loss.item(), tr_acc, va_acc, history[-1]['weight_norm']))

        if step % cfg.embed_every == 0:
            embed_steps.append(step)
            embed_mats.append(model.number_embeddings())

    # write both artifacts before returning, so a later disconnect cannot lose them
    with open(os.path.join(cfg.out_dir, 'history.json'), 'w') as f:
        json.dump({'config': asdict(cfg), 'history': history}, f)
    np.savez_compressed(os.path.join(cfg.out_dir, 'embed_log.npz'),
                        steps=np.array(embed_steps),
                        embeds=np.stack(embed_mats).astype(np.float32))

    mins = (time.time() - t0) / 60
    grokk = next((r['step'] for r in history if r['val_acc'] >= 0.95), None)
    print('  done in %.1f min | grokk step: %s' % (mins, grokk))
    return history

print('train() defined')


In [ ]:
# ── 5. Smoke test: p=31 groks in well under a minute on GPU ────────────
# Verified on CPU: groks at step 4,500 in 2.4 min. On T4 expect ~20 s.
# NOTE: p=13 at 50% train has only 84 examples, which is below the data
# threshold where grokking can happen at all (Nanda et al., Appendix C.2).
# It memorizes and never generalizes. p=31 is the smallest quick check
# that actually reproduces the phenomenon.

smoke = Config(modulus=31, train_fraction=0.5, weight_decay=1.0,
               max_steps=8000, eval_every=500, embed_every=2000,
               batch_size=128, curriculum='random', seed=0,
               out_dir='/content/smoke')
h1 = train(smoke, verbose_every=2000)

g = next((r['step'] for r in h1 if r['val_acc'] >= 0.95), None)
print('')
if g is None:
    print('DID NOT GROKK. Do not start the grid. Raise max_steps and retry.')
else:
    print('grokked at step %d (expected ~4500) — recipe is working' % g)

print('\nrepeating identical config to check determinism...')
h2 = train(smoke, verbose_every=10**9)
same = all(abs(a['val_acc'] - b['val_acc']) < 1e-9 for a, b in zip(h1, h2))
print('deterministic at fixed seed:', same)
if not same:
    print('  Runs differ at the same seed. The grid still works, but say so')
    print('  in the paper instead of claiming exact reproducibility.')


In [ ]:
# ── 6. Main grid: 5 curricula x 3 seeds, addition mod 97 ────────────────
# Writes to Drive after each run and skips finished ones, so re-running this
# cell after a disconnect resumes where it stopped.
# Expect roughly 10-20 min per run, so ~3-5 h total.

SEEDS      = [0, 1, 2]
OPERATION  = 'add'
SEP        = '=' * 62

done_n, skip_n = 0, 0
for seed in SEEDS:
    for cur in CURRICULA:
        tag = '%s_p97_%s_s%d' % (OPERATION, cur, seed)
        out_dir = os.path.join(DRIVE_DIR, tag)
        if os.path.exists(os.path.join(out_dir, 'history.json')):
            print('[skip] ' + tag)
            skip_n += 1
            continue
        print('')
        print(SEP)
        print('START  ' + tag)
        print(SEP)
        cfg = Config(modulus=97, operation=OPERATION, curriculum=cur,
                     seed=seed, out_dir=out_dir)
        train(cfg)
        done_n += 1

print('')
print(SEP)
print('grid finished: %d new, %d skipped' % (done_n, skip_n))
print('saved under ' + DRIVE_DIR)


In [ ]:
# ── 7. OPTIONAL: multiplication baseline ────────────────────────────────
# The GPT-2 paper has no mul/random baseline, which is why the mul claims are
# weak. This fills that gap. Run only if cell 6 finished with time to spare.

RUN_MUL = False   # flip to True to run

if RUN_MUL:
    for seed in [0, 1, 2]:
        for cur in ['random', 'easy_hard']:
            tag = 'mul_p97_%s_s%d' % (cur, seed)
            out_dir = os.path.join(DRIVE_DIR, tag)
            if os.path.exists(os.path.join(out_dir, 'history.json')):
                print('[skip] ' + tag)
                continue
            print('\nSTART  ' + tag)
            cfg = Config(modulus=97, operation='mul', curriculum=cur,
                         seed=seed, max_steps=50000, out_dir=out_dir)
            train(cfg)
else:
    print('skipped (set RUN_MUL = True to enable)')


In [ ]:
# ── 8. Results table ────────────────────────────────────────────────────
from scipy import stats

def grokk_step(h, thr=0.95):
    return next((r['step'] for r in h if r['val_acc'] >= thr), None)

runs = {}
for seed in SEEDS:
    for cur in CURRICULA:
        tag = 'add_p97_%s_s%d' % (cur, seed)
        pth = os.path.join(DRIVE_DIR, tag, 'history.json')
        if os.path.exists(pth):
            with open(pth) as f:
                runs[(cur, seed)] = json.load(f)['history']

print('loaded %d / %d runs\n' % (len(runs), len(SEEDS) * len(CURRICULA)))

print('%-22s %s' % ('condition', '  '.join('seed%d' % s for s in SEEDS)))
print('-' * 62)
summary = {}
for cur in CURRICULA:
    steps = []
    cells = []
    for s in SEEDS:
        h = runs.get((cur, s))
        if h is None:
            cells.append('  --  ')
            continue
        g = grokk_step(h)
        cells.append('%6s' % (g if g is not None else 'DNF'))
        if g is not None:
            steps.append(g)
    if steps and len(steps) == len(SEEDS):
        m, sd = float(np.mean(steps)), float(np.std(steps, ddof=1))
        summary[cur] = (m, sd, steps)
        tail = '  mean %.0f +/- %.0f' % (m, sd)
    elif steps:
        tail = '  (partial: %d/%d grokked)' % (len(steps), len(SEEDS))
    else:
        tail = '  no run grokked'
    print('%-22s %s%s' % (cur, '  '.join(cells), tail))

print('')
if 'random' in summary and 'easy_hard' in summary:
    r, e = summary['random'][2], summary['easy_hard'][2]
    t, p = stats.ttest_ind(r, e, equal_var=False)
    delta = (np.mean(e) - np.mean(r)) / np.mean(r) * 100
    print('random vs easy_hard: %+.0f%%   Welch t=%.2f  p=%.4f' % (delta, t, p))
    if p >= 0.05:
        print('  NOT significant at 0.05. Report it that way.')
else:
    print('need both random and easy_hard at all seeds for the t-test')


In [ ]:
# ── 9. Figures ──────────────────────────────────────────────────────────
COLORS = {'random': '#555555', 'easy_hard': '#2196F3', 'hard_easy': '#F44336',
          'residue_blocks': '#4CAF50', 'balanced_mini_epoch': '#9C27B0'}
NICE = {'random': 'Random (baseline)', 'easy_hard': 'Easy to Hard',
        'hard_easy': 'Hard to Easy', 'residue_blocks': 'Residue Blocks',
        'balanced_mini_epoch': 'Balanced Mini-Epoch'}

def band(ax, cur, metric):
    hs = [runs[(cur, s)] for s in SEEDS if (cur, s) in runs]
    if not hs:
        return
    grid = np.array([r['step'] for r in hs[0]])
    mat = np.stack([np.interp(grid, [r['step'] for r in h], [r[metric] for r in h])
                    for h in hs])
    mu, sd = mat.mean(0), mat.std(0)
    ax.plot(grid, mu, color=COLORS[cur], label=NICE[cur], linewidth=2)
    if len(hs) > 1:
        ax.fill_between(grid, mu - sd, mu + sd, color=COLORS[cur], alpha=0.15)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
fig.suptitle('From-scratch transformer, addition mod 97 (mean +/- 1 SD over seeds)',
             fontsize=12, fontweight='bold')

for ax, metric, ylab in zip(
        axes,
        ['val_acc', 'gen_gap', 'weight_norm'],
        ['Validation accuracy', 'Generalization gap', 'Weight norm']):
    for cur in CURRICULA:
        band(ax, cur, metric)
    ax.set_xlabel('Training step')
    ax.set_ylabel(ylab)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7)
    if metric == 'val_acc':
        ax.axhline(0.95, color='gray', linestyle=':', alpha=0.7)
        ax.set_ylim(-0.05, 1.05)

axes[2].set_title('does the rise-then-fall appear here?', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'scratch_overview.png'), dpi=180, bbox_inches='tight')
plt.show()

# The key comparison against the GPT-2 result
print('')
print('WEIGHT NORM: REPLICATION CHECK')
print('-' * 62)
print('Nanda et al. (2023, Fig. 1 bottom-right) report that the sum of squared')
print('weights DECREASES smoothly during circuit formation and more sharply')
print('during cleanup. Both phases are driven by weight decay. They do NOT')
print('report a rise-then-fall, and weight norm is not one of their proposed')
print('progress measures. So a monotonic decrease here is a REPLICATION,')
print('not a contradiction. Do not write it up as a novel negative result.')
print('')
for cur in CURRICULA:
    for s_ in SEEDS:
        h = runs.get((cur, s_))
        if h is None:
            continue
        wn = [r['weight_norm'] for r in h]
        pk = h[int(np.argmax(wn))]['step']
        mono = all(wn[i+1] <= wn[i] + 1e-6 for i in range(len(wn)-1))
        print('  %-22s s%d  start %7.1f  end %7.1f  peak@%-6d %s'
              % (cur, s_, wn[0], wn[-1], pk,
                 'monotonic' if mono else 'non-monotonic'))


In [ ]:
# ── 10. Fourier structure of the embeddings ─────────────────────────────
# Nanda et al. found that grokking coincides with a sparse set of frequencies
# emerging in the embedding matrix. Question here: do random and easy_hard
# converge on the SAME frequencies at different times, or different ones?

def spectra_for(cur, seed=0):
    pth = os.path.join(DRIVE_DIR, 'add_p97_%s_s%d' % (cur, seed), 'embed_log.npz')
    if not os.path.exists(pth):
        return None, None
    z = np.load(pth)
    steps, emb = z['steps'], z['embeds']          # emb: (n, 97, d_model)
    mag = np.abs(np.fft.rfft(emb, axis=1))        # (n, 49, d_model)
    return steps, mag.mean(axis=2)                # (n, 49)

PAIR = ['random', 'easy_hard']
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('Fourier spectrum of number embeddings', fontsize=12, fontweight='bold')

for col, cur in enumerate(PAIR):
    steps, spec = spectra_for(cur)
    if steps is None:
        axes[0, col].set_title(NICE[cur] + ': no embed_log.npz')
        continue
    h = runs.get((cur, 0))
    g = grokk_step(h) if h else None

    ax = axes[0, col]
    im = ax.imshow(spec.T, aspect='auto', origin='lower', cmap='magma',
                   extent=[steps[0], steps[-1], 0, spec.shape[1]])
    if g:
        ax.axvline(g, color='cyan', linestyle='--', linewidth=1.5)
    ax.set_title('%s  (grokk at %s)' % (NICE[cur], g))
    ax.set_xlabel('Training step')
    ax.set_ylabel('Frequency')
    plt.colorbar(im, ax=ax, label='|DFT|')

    ax = axes[1, col]
    final = spec[-1]
    top = np.argsort(final)[::-1][:5]
    ax.bar(np.arange(len(final)), final, color=COLORS[cur], alpha=0.85)
    ax.set_title('final spectrum, top freqs: %s' % sorted(top.tolist()))
    ax.set_xlabel('Frequency')
    ax.set_ylabel('|DFT|')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'fourier.png'), dpi=180, bbox_inches='tight')
plt.show()

print('')
print('DOMINANT FREQUENCIES AT END OF TRAINING')
print('-' * 62)
tops = {}
for cur in PAIR:
    steps, spec = spectra_for(cur)
    if steps is None:
        continue
    tops[cur] = set(np.argsort(spec[-1])[::-1][:5].tolist())
    print('  %-22s %s' % (cur, sorted(tops[cur])))
if len(tops) == 2:
    a, b = tops[PAIR[0]], tops[PAIR[1]]
    shared = sorted(a & b)
    print('')
    print('  shared: %s  (%d of 5)' % (shared, len(shared)))
    if len(shared) >= 4:
        print('  -> same representation, reached at different times.')
        print('     Supports: curriculum changes timing, not mechanism.')
    elif len(shared) <= 2:
        print('  -> different representations. Stronger claim, needs more seeds')
        print('     before you write it down.')
    else:
        print('  -> partial overlap. Ambiguous; check more seeds.')


In [ ]:
# ── 10b. Gini coefficient of Fourier norms (Nanda's real progress measure)
# Nanda et al. report this rising sharply during cleanup, as the embedding
# concentrates onto a sparse set of key frequencies. It is computable from
# the embed_log we already save, and unlike weight norm it is actually what
# the paper proposes. If it spikes before the val_acc jump, it is a genuine
# progress measure and the curriculum question becomes: does easy_hard
# delay the spike, or delay the jump that follows it?

def gini(x):
    x = np.sort(np.abs(np.asarray(x, dtype=np.float64)))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float(((2 * idx - n - 1) * x).sum() / (n * x.sum()))

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.set_title('Gini coefficient of Fourier component norms (higher = sparser)',
             fontsize=11, fontweight='bold')

for cur in PAIR:
    steps, spec = spectra_for(cur)
    if steps is None:
        continue
    g_series = [gini(spec[i]) for i in range(len(steps))]
    ax.plot(steps, g_series, color=COLORS[cur], label=NICE[cur], linewidth=2)
    h = runs.get((cur, 0))
    gs = grokk_step(h) if h else None
    if gs:
        ax.axvline(gs, color=COLORS[cur], linestyle='--', alpha=0.6)
        near = int(np.argmin(np.abs(steps - gs)))
        rise = next((int(steps[i]) for i in range(1, len(g_series))
                     if g_series[i] - g_series[0] > 0.5 * (g_series[near] - g_series[0])),
                    None)
        if rise is not None:
            print('%-12s gini half-rise at step %-6s | grokk at %-6s | lead %s steps'
                  % (cur, rise, gs, gs - rise))

ax.set_xlabel('Training step')
ax.set_ylabel('Gini coefficient')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'gini.png'), dpi=180, bbox_inches='tight')
plt.show()
print('')
print('Positive lead = Gini rises BEFORE grokking, i.e. it predicts the jump.')
print('This is the progress-measure result worth reporting. Weight norm is not.')


In [ ]:
# ── 11. Paper-ready output ──────────────────────────────────────────────
# Prints a LaTeX table and a numbers.json holding every value the paper cites,
# so nothing has to be retyped or remembered.

rows = []
for cur in CURRICULA:
    cells = []
    for s in SEEDS:
        h = runs.get((cur, s))
        if h is None:
            cells.append('--')
        else:
            g = grokk_step(h)
            cells.append('{:,}'.format(g) if g else 'DNF')
    if cur in summary:
        m, sd, _ = summary[cur]
        stat = '${:,.0f} \\pm {:,.0f}$'.format(m, sd)
    else:
        stat = 'n/a'
    rows.append('%s & %s & %s \\\\' % (NICE[cur], ' & '.join(cells), stat))

print('% --- paste into the paper ---')
print('\\begin{table}[H]')
print('\\centering')
print('\\caption{From-scratch transformer, addition mod 97. Step at which validation')
print('accuracy first reaches 95\\%. DNF means no grokking within the step budget.}')
print('\\small')
print('\\begin{tabular}{@{}l' + 'c' * len(SEEDS) + 'c@{}}')
print('\\toprule')
print('\\textbf{Condition} & ' +
      ' & '.join('\\textbf{Seed %d}' % s for s in SEEDS) +
      ' & \\textbf{Mean $\\pm$ SD} \\\\')
print('\\midrule')
for r in rows:
    print(r)
print('\\bottomrule')
print('\\end{tabular}')
print('\\end{table}')

export = {
    'setup': {'model': '2-layer transformer from scratch', 'd_model': 128,
              'n_heads': 4, 'params': 'see cell 2', 'modulus': 97,
              'operation': 'add', 'seeds': SEEDS, 'threshold': 0.95},
    'grokking_steps': {cur: {str(s): grokk_step(runs[(cur, s)])
                             for s in SEEDS if (cur, s) in runs}
                       for cur in CURRICULA},
    'mean_sd': {cur: {'mean': summary[cur][0], 'sd': summary[cur][1]}
                for cur in summary},
    'weight_norm_peak_step': {cur: {str(s): int(runs[(cur, s)][int(np.argmax(
                                  [r['weight_norm'] for r in runs[(cur, s)]]))]['step'])
                                    for s in SEEDS if (cur, s) in runs}
                              for cur in CURRICULA},
}
if 'random' in summary and 'easy_hard' in summary:
    t, p = stats.ttest_ind(summary['random'][2], summary['easy_hard'][2], equal_var=False)
    export['welch_t'] = {'t': float(t), 'p': float(p)}

with open(os.path.join(DRIVE_DIR, 'numbers.json'), 'w') as f:
    json.dump(export, f, indent=2)
print('')
print('wrote ' + os.path.join(DRIVE_DIR, 'numbers.json'))
print('Every number the paper cites is in that file. Cite from it, not from memory.')
